# 1: Instalasi Library

In [1]:
# Install library yang dibutuhkan[cite: 1]
!pip install -q streamlit pyngrok google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 61.7 MB/s eta 0:00:00


# 2: Konfigurasi Ngrok & Fungsi Helper

In [2]:
# Setup Ngrok dan fungsi untuk menjalankan Streamlit di background[cite: 1]
from pyngrok import ngrok
from google.colab import userdata
import subprocess
import time

# Masukkan NGROK_TOKEN di Colab Secrets[cite: 1]
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

def run_streamlit(filename, port=8501):
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    ngrok.kill()
    time.sleep(3)

    proc = subprocess.Popen(
        ["streamlit", "run", filename, "--server.headless=true", "--server.port", str(port), "--server.enableCORS=false"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Aplikasi IT Helpdesk berjalan di: {public_url}")
    return proc

# 3: Kode Utama Aplikasi IT Helpdesk (Streamlit)

In [6]:
%%writefile it_helpdesk_app.py
import streamlit as st
from google import genai
from google.colab import userdata

# Konfigurasi UI awal[cite: 1]
st.title("💻 IT Helpdesk Assistant")
st.caption("Asisten virtual santai & ramah untuk masalah komputer, internet, dan IT.")

# Sidebar konfigurasi[cite: 1]
with st.sidebar:
    google_api_key = st.text_input("Google AI API Key", type="password")
    reset_button = st.button("Reset Percakapan")

if not google_api_key:
    st.info("Masukkan API Key Google AI untuk memulai.")
    st.stop()

# Inisialisasi client & session[cite: 1]
if ("genai_client" not in st.session_state) or (getattr(st.session_state, "_last_key", None) != google_api_key):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"Error: {e}")
        st.stop()

# Set persona IT Support yang santai dan ramah
if "chat" not in st.session_state:
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash",
        config={
            "system_instruction": "Kamu adalah asisten IT Helpdesk/Support virtual. Tugasmu membantu user mengatasi masalah komputer, internet, jaringan, dan IT secara umum. Gunakan gaya bahasa yang santai, ramah, dan personal layaknya teman yang jago IT. Hindari bahasa yang terlalu kaku atau teknis tanpa penjelasan."
        }
    )

if "messages" not in st.session_state:
    st.session_state.messages = []

# Reset chat
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.rerun()

# Render riwayat chat[cite: 1]
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Input dan pemrosesan pesan[cite: 1]
prompt = st.chat_input("Ceritain masalah IT kamu di sini...")

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    try:
        response = st.session_state.chat.send_message(prompt)
        answer = response.text if hasattr(response, "text") else str(response)
    except Exception as e:
        answer = f"Waduh, ada error nih: {e}"

    with st.chat_message("assistant"):
        st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting it_helpdesk_app.py


# 4: Menjalankan Aplikasi

In [4]:
# Jalankan file aplikasi dan buka tunnel ngrok[cite: 1]
proc = run_streamlit("it_helpdesk_app.py")

Aplikasi IT Helpdesk berjalan di: NgrokTunnel: "https://e603-34-132-251-189.ngrok-free.app" -> "http://localhost:8501"


In [7]:
# Hentikan proses jika sudah selesai[cite: 1]
try:
    proc.terminate()
except:
    pass
ngrok.kill()
print("Aplikasi dan tunnel berhasil dimatikan.")

Aplikasi dan tunnel berhasil dimatikan.
